In [2]:
!pip install xgboost

In [3]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [5]:
df = pd.read_excel("DF_FINAL.xlsx")

df.head()

calles_base = [
        "Calle Alberto Aguilera",
        "Calle Atocha",
        "Calle Gran Vía",
        "Calle Hortaleza",
        "Calle Mayor",
        "Calle Princesa",
        "Paseo Infanta Isabel",
        "Paseo de El Prado"
]

calles_oeste = []

for base in calles_base:
    for col in df.columns:
        if base in col:
            calles_oeste.append(col)

L = 3  # número de lags

X = pd.DataFrame()

for street in calles_oeste:
    for lag in range(1, L + 1):
        X[f"{street}_t-{lag}"] = df[street].shift(lag)

# Variables externas reales
X["AEMET_tmed"] = df["AEMET_tmed"]
X["AEMET_prec"] = df["AEMET_prec"]
X["dia_semana"] = df["dia_semana"]
X["festivo"] = df["festivo"]

# Convertir variables categóricas a numéricas
X["dia_semana"] = X["dia_semana"].map({
    "lunes": 0,
    "martes": 1,
    "miércoles": 2,
    "jueves": 3,
    "viernes": 4,
    "sábado": 5,
    "domingo": 6
})

X["festivo"] = X["festivo"].map({
    "no": 0,
    "sí": 1
})


y_col = [col for col in df.columns if "Calle Alberto Aguilera" in col][0]
y = df[y_col]

# Limpiar NaNs
X = X.dropna()
y = y.loc[X.index]

In [6]:
# División temporal (80% train, 20% test)
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(X_train.shape, X_test.shape)

(15376, 43) (3845, 43)


In [7]:
with pd.ExcelWriter("dataset_XGBoost_AlbertoAguilera.xlsx") as writer:
    X_train.to_excel(writer, sheet_name="X_train")
    X_test.to_excel(writer, sheet_name="X_test")
    y_train.to_excel(writer, sheet_name="y_train")
    y_test.to_excel(writer, sheet_name="y_test")

In [12]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

In [18]:
y_pred = model.predict(X_test)
# Predicción próxima hora
X_next = X.iloc[[-1]]
y_next = model.predict(X_next)

print("Predicción tráfico en Alberto Aguilera (E-O) próxima hora:", y_next[0])

Predicción tráfico en Alberto Aguilera (E-O) próxima hora: 363.436


In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

MAE: 88.09
RMSE: 184.95
MAPE: 10.81%


In [20]:
importances = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances.head(10))

Calle Alberto Aguilera_O-E_t-1    0.609733
Calle Alberto Aguilera_E-O_t-1    0.190120
Paseo de El Prado_S-N_t-1         0.018732
Paseo de El Prado_S-N_t-3         0.018505
Calle Atocha_E-O_t-3              0.013035
Paseo Infanta Isabel_E-O_t-3      0.010951
Calle Alberto Aguilera_O-E_t-3    0.009878
Calle Princesa_S-N_t-3            0.008918
Calle Mayor_O-E_t-3               0.008532
Calle Alberto Aguilera_O-E_t-2    0.007320
dtype: float32


In [22]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=0,
    n_jobs=-1,
    scoring="neg_mean_absolute_error"
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

perm_df.head(10)


,feature,importance
3,Calle Alberto Aguilera_O-E_t-1,220.871986
0,Calle Alberto Aguilera_E-O_t-1,182.461571
4,Calle Alberto Aguilera_O-E_t-2,8.153609
38,Paseo de El Prado_S-N_t-3,6.026996
5,Calle Alberto Aguilera_O-E_t-3,5.380982
29,Paseo Infanta Isabel_E-O_t-3,4.194756
36,Paseo de El Prado_S-N_t-1,3.866206
2,Calle Alberto Aguilera_E-O_t-3,3.624027
41,dia_semana,2.916563
18,Calle Mayor_O-E_t-1,2.840607


In [23]:
top_k = 8
top_features = perm_df.head(top_k)
top_features


,feature,importance
3,Calle Alberto Aguilera_O-E_t-1,220.871986
0,Calle Alberto Aguilera_E-O_t-1,182.461571
4,Calle Alberto Aguilera_O-E_t-2,8.153609
38,Paseo de El Prado_S-N_t-3,6.026996
5,Calle Alberto Aguilera_O-E_t-3,5.380982
29,Paseo Infanta Isabel_E-O_t-3,4.194756
36,Paseo de El Prado_S-N_t-1,3.866206
2,Calle Alberto Aguilera_E-O_t-3,3.624027


In [24]:
top_streets = (
    top_features["feature"]
    .str.split("_")
    .str[0]
    .value_counts()
)
top_streets

feature
Calle Alberto Aguilera    5
Paseo de El Prado         2
Paseo Infanta Isabel      1
Name: count, dtype: int64